<a href="https://colab.research.google.com/github/Sciform/sciform-hwz-ai-in-controlling/blob/main/lecture_2/2_8_ts_neural_prophet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Global Time Series Model with NeuralProphet – Sales Forecasting for an E-Commerce Assortment

In the previous notebooks we trained **one model per time series** (a *local* model). In retail, however, we usually have to forecast
hundreds or thousands of products at once. A **global model** is trained on *all* time series of the assortment at the same time:

* the model shares its parameters (trend, seasonality, holiday effects, autoregression) across products,
* products with short or noisy histories profit from the patterns learned on the other products,
* only **one** model has to be trained, monitored and deployed.

[NeuralProphet](https://neuralprophet.com) is a PyTorch-based successor of Facebook Prophet. It supports global modelling out of the box:
the data frame just needs an additional column `ID` that identifies the time series. With the parameters
`trend_global_local` and `season_global_local` we decide whether trend and seasonality are shared (`"global"`) or estimated per series (`"local"`).

**Agenda**

1. Load the *Online Retail* data set (UK online gift shop, Dec 2010 – Dec 2011)
2. Create sample data: daily revenue for an assortment of the 10 most regularly sold products
3. Baseline: seasonal naive forecast
4. Local vs. global NeuralProphet model
5. Global model with autoregression (AR-Net)
6. Evaluation per product and for the whole assortment (controlling view)
7. Final forecast for the next 4 weeks

## Setup

In [ ]:
# Install the required packages (NeuralProphet is not pre-installed on Colab)
%pip install -q pandas "numpy<2" matplotlib neuralprophet

# NeuralProphet needs numpy < 2, Colab ships numpy 2.x -> pip downgrades numpy.
# The new version is only used after a restart of the Python session.
# This cell restarts the session automatically if necessary:
# after the restart, simply continue with the NEXT cell.
import os, importlib.metadata
import numpy
if numpy.__version__ != importlib.metadata.version("numpy"):
    print("Packages were updated - restarting the session. Afterwards continue with the next cell.")
    os.kill(os.getpid(), 9)
else:
    print("numpy", numpy.__version__, "- ready, continue with the next cell.")

In [ ]:
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from neuralprophet import NeuralProphet, set_log_level, set_random_seed

warnings.filterwarnings("ignore")
set_log_level("ERROR")                      # keep the training output short
logging.getLogger("lightning").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Load the data

The data set contains all transactions of a UK-based online retailer selling unique all-occasion gifts. Many customers are wholesalers.

| column | meaning |
|---|---|
| InvoiceNo | invoice number, starts with `C` for cancellations |
| StockCode | product code |
| Description | product name |
| Quantity | quantity per transaction line |
| InvoiceDate | date and time of the invoice |
| UnitPrice | price per unit in GBP |
| CustomerID | customer number |
| Country | country of the customer |

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/sciform/sciform-hwz-ai-in-controlling/main/data/online_retail.csv

In [ ]:
raw = pd.read_csv("online_retail.csv", dtype={"InvoiceNo": str, "StockCode": str}, encoding="ISO-8859-1")
raw["InvoiceDate"] = pd.to_datetime(raw["InvoiceDate"], format="%m/%d/%Y %H:%M")
print(raw.shape)
raw.head()

### Data cleaning

* remove cancellations (`InvoiceNo` starting with `C`) and lines with non-positive quantity or price,
* remove non-product positions such as postage (`POST`), manual corrections (`M`), bank charges, Amazon fees, ... –
  real products have a stock code starting with five digits,
* compute the **revenue** per line: `Sales = Quantity * UnitPrice`.

In [ ]:
df = raw[~raw["InvoiceNo"].str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df = df[df["StockCode"].str.match(r"^\d{5}")]
df = df.assign(Sales=df["Quantity"] * df["UnitPrice"], Date=df["InvoiceDate"].dt.normalize())

print(f"{len(df):,} transaction lines, {df['StockCode'].nunique():,} products, "
      f"{df['Date'].min().date()} – {df['Date'].max().date()}")

## 2. Create sample data: the assortment

Most of the ~3,900 products are sold only sporadically. For this example we select the **10 products that are sold on the most days**
(regular sellers – "Schnelldreher"). This is our assortment. For each product we build a **daily revenue time series**:

* days without sales (e.g. Saturdays – the shop does not ship on Saturdays – and the Christmas break) get revenue 0,
* single bulk orders of wholesalers create extreme peaks. We cap each series at its 99 % quantile so that these one-off events do not
  dominate the training (in practice they would be planned separately, e.g. as key-account orders),
* the result is stored in *long format* with the columns `ds` (date), `ID` (product) and `y` (revenue) – exactly the format that
  NeuralProphet expects for a global model.

In [ ]:
N_PRODUCTS = 10

selling_days = df.groupby("StockCode")["Date"].nunique().sort_values(ascending=False)
assortment = selling_days.head(N_PRODUCTS).index

# most frequent description per stock code
names = (df[df["StockCode"].isin(assortment)]
         .groupby("StockCode")["Description"].agg(lambda s: s.mode().iat[0]))

overview = pd.DataFrame({
    "Description": names,
    "SellingDays": selling_days[assortment],
    "Revenue_GBP": df[df["StockCode"].isin(assortment)].groupby("StockCode")["Sales"].sum(),
}).loc[assortment]
overview

In [ ]:
daily = (df[df["StockCode"].isin(assortment)]
         .groupby(["StockCode", "Date"])["Sales"].sum()
         .unstack("StockCode"))

# complete calendar: days without sales get revenue 0
full_range = pd.date_range(df["Date"].min(), df["Date"].max(), freq="D")
daily = daily.reindex(full_range).fillna(0.0)

# cap one-off bulk orders at the 99% quantile of each product
daily = daily.clip(upper=daily.quantile(0.99), axis=1)

# long format for NeuralProphet: ds | ID | y
sales = (daily.rename_axis("ds").reset_index()
         .melt(id_vars="ds", var_name="ID", value_name="y")
         .sort_values(["ID", "ds"]).reset_index(drop=True))

sales.to_csv("online_retail_assortment_daily_sales.csv", index=False)   # sample data for later use
print(sales.shape)
sales.head()

In [ ]:
def plot_grid(frames, title, ncols=2):
    # frames: dict ID -> callable(ax)
    ids = list(frames)
    nrows = int(np.ceil(len(ids) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 2.6 * nrows), sharex=True)
    for ax, i in zip(axes.flat, ids):
        frames[i](ax)
        ax.set_title(f"{i} – {names[i][:35]}", fontsize=10)
        ax.grid(alpha=0.3)
    for ax in axes.flat[len(ids):]:
        ax.set_visible(False)
    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    plt.show()

weekly = daily.resample("W").sum()
plot_grid({i: (lambda ax, i=i: ax.plot(weekly.index, weekly[i], color="tab:blue")) for i in assortment},
          "Weekly revenue per product (GBP) – sample data")

In [ ]:
# day-of-week pattern of the whole assortment (Mon=0 ... Sun=6) – note: no sales on Saturdays
daily.sum(axis=1).groupby(daily.index.dayofweek).mean().plot.bar(
    title="Average daily revenue of the assortment by weekday", xlabel="weekday (0 = Monday)", ylabel="GBP");

### Train / test split

We forecast the last **4 weeks (28 days)** of each product. All earlier data is used for training.
Because all series share the same calendar, a simple date cut-off gives the same split for every product.

> We have only ~1 year of data, so a yearly seasonality cannot be estimated reliably and is switched off.
> The pre-Christmas peak in Nov/Dec is partly absorbed by the trend.

In [ ]:
HORIZON = 28
cutoff = sales["ds"].max() - pd.Timedelta(days=HORIZON)

df_train = sales[sales["ds"] <= cutoff].reset_index(drop=True)
df_test = sales[sales["ds"] > cutoff].reset_index(drop=True)
print(f"train: {df_train['ds'].min().date()} – {df_train['ds'].max().date()}  ({df_train['ds'].nunique()} days)")
print(f"test:  {df_test['ds'].min().date()} – {df_test['ds'].max().date()}  ({df_test['ds'].nunique()} days)")

### Error metrics

* **MAE** – mean absolute error in GBP per day,
* **RMSE** – root mean squared error, punishes large errors,
* **WAPE** – weighted absolute percentage error = Σ|y − ŷ| / Σ|y|. Unlike MAPE it also works with days of zero revenue
  and can be compared across products of different size – a common KPI in retail demand planning.

In [ ]:
def metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    err = y_true - y_pred
    return pd.Series({"MAE": np.mean(np.abs(err)),
                      "RMSE": np.sqrt(np.mean(err ** 2)),
                      "WAPE_%": 100 * np.sum(np.abs(err)) / np.sum(np.abs(y_true))})

def evaluate(pred, col="yhat", name="model"):
    # pred: data frame with ds, ID, y, <col>. Returns metrics per product + over all products.
    per_id = pred.groupby("ID").apply(lambda g: metrics(g["y"], g[col]))
    per_id.loc["ALL"] = metrics(pred["y"], pred[col])
    per_id.columns = pd.MultiIndex.from_product([[name], per_id.columns])
    return per_id

results = {}

## 3. Baseline: seasonal naive forecast

Every model has to beat a simple benchmark. The *seasonal naive* forecast repeats the revenue of the last observed week
for all four weeks of the horizon (same weekday → same value).

In [ ]:
last_week = df_train[df_train["ds"] > cutoff - pd.Timedelta(days=7)].assign(dow=lambda d: d["ds"].dt.dayofweek)
pred_naive = (df_test.assign(dow=lambda d: d["ds"].dt.dayofweek)
              .merge(last_week[["ID", "dow", "y"]].rename(columns={"y": "yhat"}), on=["ID", "dow"]))

results["seasonal_naive"] = evaluate(pred_naive, name="seasonal_naive")
results["seasonal_naive"].loc[["ALL"]]

## 4. Local vs. global NeuralProphet model

NeuralProphet decomposes the time series into interpretable components:

$$\hat y_t = \text{trend}(t) + \text{seasonality}(t) + \text{events}(t) + \text{AR}(t) + \dots$$

We use

* a piecewise linear **trend** with a few change points,
* a **weekly seasonality** (strong weekday effect, no shipping on Saturdays),
* **UK public holidays** as events,
* `quantiles=[0.1, 0.9]` to obtain an 80 % prediction interval in addition to the point forecast.

By default NeuralProphet normalises each time series separately (`global_normalization=False`). This is important for a global model,
because the products have very different revenue levels – the model learns the common *shape* on the normalised scale.

The only difference between the two models below is the parameter `..._global_local`:

| | `trend_global_local` | `season_global_local` | number of models / parameter sets |
|---|---|---|---|
| local model | `"local"` | `"local"` | one trend + seasonality **per product** |
| global model | `"global"` | `"global"` | **one** trend + seasonality shared by all products |

In [ ]:
def build_model(global_local="global", n_lags=0, n_forecasts=1, epochs=60):
    set_random_seed(42)
    m = NeuralProphet(
        trend_global_local=global_local,
        season_global_local=global_local,
        n_changepoints=5,
        yearly_seasonality=False,      # only one year of history
        weekly_seasonality=True,
        daily_seasonality=False,
        n_lags=n_lags,                 # > 0 activates the autoregression (AR-Net)
        n_forecasts=n_forecasts,
        quantiles=[0.1, 0.9],
        epochs=epochs,
        batch_size=64,
        learning_rate=0.01,
    )
    m.add_country_holidays("GB")
    m.set_plotting_backend("matplotlib")
    return m

def trainer_kwargs():
    # quiet training without progress bars
    return dict(freq="D", progress=None, minimal=True)

In [ ]:
%%time
m_local = build_model("local")
m_local.fit(df_train, **trainer_kwargs())

future = m_local.make_future_dataframe(df_train, periods=HORIZON, n_historic_predictions=False)
fc_local = m_local.predict(future)
pred_local = df_test.merge(fc_local[["ds", "ID", "yhat1"]], on=["ds", "ID"]).rename(columns={"yhat1": "yhat"})

results["np_local"] = evaluate(pred_local, name="np_local")
results["np_local"].loc[["ALL"]]

In [ ]:
%%time
m_global = build_model("global")
m_global.fit(df_train, **trainer_kwargs())

future = m_global.make_future_dataframe(df_train, periods=HORIZON, n_historic_predictions=False)
fc_global = m_global.predict(future)
pred_global = df_test.merge(fc_global[["ds", "ID", "yhat1", "yhat1 10.0%", "yhat1 90.0%"]], on=["ds", "ID"]) \
                     .rename(columns={"yhat1": "yhat", "yhat1 10.0%": "lo", "yhat1 90.0%": "hi"})

results["np_global"] = evaluate(pred_global, name="np_global")
results["np_global"].loc[["ALL"]]

### Components of the global model

Because trend and seasonality are global, there is just one weekly profile for the whole assortment (on the normalised scale).

In [ ]:
m_global.plot_parameters(df_name=assortment[0])
plt.show()

## 5. Global model with autoregression (AR-Net)

Trend and seasonality only depend on the calendar. With `n_lags > 0` NeuralProphet additionally uses the **last observed values** of each
series as inputs (autoregression). With `n_forecasts = 28` the model predicts the complete horizon at once from the last 28 days
(*multi-step forecast*): output `yhat1` is the forecast one day ahead, `yhat28` the forecast 28 days ahead.

The AR weights are shared by all products as well – this is where a global model shines: a local AR model with 28 lags
would have only ~300 training samples per product, the global model sees 10× more.

In [ ]:
%%time
N_LAGS = 28
m_ar = build_model("global", n_lags=N_LAGS, n_forecasts=HORIZON, epochs=60)
m_ar.fit(df_train, **trainer_kwargs())

# the future data frame contains the last N_LAGS days of history as model input + HORIZON empty days
future = m_ar.make_future_dataframe(df_train, periods=HORIZON, n_historic_predictions=False)
fc_ar = m_ar.predict(future)

# forecast origin = last training day -> on day k after the origin the forecast is in column yhat<k>
fc_ar = fc_ar[fc_ar["ds"] > cutoff].copy()
fc_ar["step"] = (fc_ar["ds"] - cutoff).dt.days
fc_ar["yhat"] = [row[f"yhat{k}"] for k, (_, row) in zip(fc_ar["step"], fc_ar.iterrows())]
pred_ar = df_test.merge(fc_ar[["ds", "ID", "yhat"]], on=["ds", "ID"])

results["np_global_ar"] = evaluate(pred_ar, name="np_global_ar")
results["np_global_ar"].loc[["ALL"]]

## 6. Evaluation

### Overall comparison (all products, 28-day horizon)

In [ ]:
comparison = pd.concat(results.values(), axis=1)
overall = comparison.loc["ALL"].unstack(0).T
overall

In [ ]:
overall["WAPE_%"].plot.barh(title="WAPE over the whole assortment (lower is better)", xlabel="WAPE in %");

In [ ]:
# store the test forecasts – notebook 2_9 compares them with a GenAI foundation model (Chronos)
test_predictions = (pred_global[["ds", "ID", "y", "yhat"]].rename(columns={"yhat": "np_global"})
                    .merge(pred_local[["ds", "ID", "yhat"]].rename(columns={"yhat": "np_local"}), on=["ds", "ID"])
                    .merge(pred_ar[["ds", "ID", "yhat"]].rename(columns={"yhat": "np_global_ar"}), on=["ds", "ID"]))
test_predictions.to_csv("neural_prophet_test_predictions.csv", index=False)
test_predictions.head()

### Accuracy per product (WAPE in %)

In [ ]:
wape = comparison.xs("WAPE_%", axis=1, level=1)
wape.insert(0, "Description", names.reindex(wape.index).fillna("all products"))
wape.style.format(precision=1, subset=wape.columns[1:]).highlight_min(axis=1, subset=wape.columns[1:], color="lightgreen")

### Forecast vs. actuals per product (global model with 80 % prediction interval)

In [ ]:
hist = df_train[df_train["ds"] > cutoff - pd.Timedelta(days=42)]

def plot_product(ax, i):
    h, p, a = hist[hist["ID"] == i], pred_global[pred_global["ID"] == i], pred_ar[pred_ar["ID"] == i]
    ax.plot(h["ds"], h["y"], color="grey", lw=1, label="history")
    ax.plot(p["ds"], p["y"], color="black", lw=1.2, label="actual")
    ax.plot(p["ds"], p["yhat"], color="tab:blue", label="global")
    ax.fill_between(p["ds"], p["lo"].clip(lower=0), p["hi"], color="tab:blue", alpha=0.2, label="80% interval")
    ax.plot(a["ds"], a["yhat"], color="tab:orange", ls="--", label="global + AR")
    ax.axvline(cutoff, color="red", lw=0.8)
    if i == assortment[0]:
        ax.legend(fontsize=8, loc="upper left")

plot_grid({i: (lambda ax, i=i: plot_product(ax, i)) for i in assortment}, "Daily revenue: forecast vs. actual (GBP)")

### Controlling view: revenue of the whole assortment

For budgeting we are often less interested in the single product than in the total. Product forecasts are summed up
(*bottom-up* forecast). Errors of single products partly cancel out, therefore the error of the total is much smaller.

In [ ]:
total = (pred_global[["ds", "y", "yhat"]].groupby("ds").sum()
         .join(pred_ar.groupby("ds")["yhat"].sum().rename("yhat_ar"))
         .join(pred_naive.groupby("ds")["yhat"].sum().rename("yhat_naive")))

ax = total[["y", "yhat", "yhat_ar", "yhat_naive"]].plot(figsize=(14, 4), style=["k-", "b-", "C1--", "g:"],
                                                       title="Daily revenue of the assortment (GBP)")
ax.legend(["actual", "global", "global + AR", "seasonal naive"]); ax.grid(alpha=0.3)
plt.show()

summary = pd.DataFrame({
    "actual_GBP": [total["y"].sum()] * 3,
    "forecast_GBP": [total["yhat"].sum(), total["yhat_ar"].sum(), total["yhat_naive"].sum()],
}, index=["np_global", "np_global_ar", "seasonal_naive"])
summary["deviation_%"] = 100 * (summary["forecast_GBP"] / summary["actual_GBP"] - 1)
summary["daily_WAPE_%"] = [metrics(total["y"], total[c])["WAPE_%"] for c in ["yhat", "yhat_ar", "yhat_naive"]]
summary

### Interpretation

* Per product, the NeuralProphet models clearly beat the seasonal naive baseline; the **global** model is at least as good as the
  local one although it has only a tenth of the trend/seasonality parameters.
* On this small assortment the autoregression brings little extra accuracy – the daily revenue of a single product is very noisy
  (a few large orders per day), so the last 28 days contain little additional information. AR becomes more valuable with
  aggregated (e.g. weekly) data or more products.
* Look at the **total revenue (deviation_%)**: the test period is the pre-Christmas peak (Nov/Dec). Without yearly seasonality the model
  cannot know about this peak and **underestimates the total**, while the seasonal naive forecast already "sees" the high level of
  the last week. With more than one year of history `yearly_seasonality=True` would fix this – a typical trade-off a controller
  must be aware of: a lower daily error does not automatically mean a better *volume* forecast.
* The weekly seasonality is smooth (Fourier terms), so the forecast for Saturdays is small but not exactly 0. A simple business rule
  (`forecast = 0 on Saturdays`) would further improve the accuracy.

## 7. Final forecast for the next 4 weeks

For production use the selected model is retrained on **all** available data and forecasts the next 28 days for every product of the assortment.
The result can be exported e.g. to a planning or BI tool.

> Note: the data ends on 9 Dec 2011, so the forecast covers the Christmas break, in which the shop hardly sells anything. The model knows the
> UK public holidays but has seen only one Christmas season – a planner would correct this period manually.

In [ ]:
%%time
m_final = build_model("global")
m_final.fit(sales, **trainer_kwargs())

future = m_final.make_future_dataframe(sales, periods=HORIZON, n_historic_predictions=False)
forecast = m_final.predict(future)[["ds", "ID", "yhat1", "yhat1 10.0%", "yhat1 90.0%"]]
forecast.columns = ["ds", "ID", "forecast", "lower_80", "upper_80"]
forecast[["forecast", "lower_80", "upper_80"]] = forecast[["forecast", "lower_80", "upper_80"]].clip(lower=0)  # no negative revenue
forecast["Description"] = forecast["ID"].map(names)

forecast.to_csv("assortment_sales_forecast.csv", index=False)
forecast.head(10)

In [ ]:
# planned revenue per product and week
plan = forecast.assign(week=forecast["ds"].dt.to_period("W")).pivot_table(
    index=["ID", "Description"], columns="week", values="forecast", aggfunc="sum")
plan["total"] = plan.sum(axis=1)
plan.loc[("ALL", "assortment"), :] = plan.sum()
plan.round(0)

## Summary and exercises

* A **global model** is trained once on all time series of the assortment. With NeuralProphet this only requires an `ID` column and
  the parameters `trend_global_local` / `season_global_local`.
* Per-series normalisation lets products of very different size share one model.
* Autoregression (`n_lags`) lets the model react to the most recent sales level; with `n_forecasts` it predicts the whole horizon at once.
* For controlling, the bottom-up aggregation of product forecasts has a much lower daily error than the single products –
  but check the bias of the total: missing seasonal effects (here: Christmas) accumulate over the assortment.

**Exercises**

1. Increase the assortment to `N_PRODUCTS = 50`. How do the local and the global model compare now? How long does training take?
2. Forecast `Quantity` (units) instead of revenue – which is the better target for replenishment, which for budgeting?
3. Try a *mixed* model: global trend but local seasonality (`trend_global_local="global"`, `season_global_local="local"`).
4. Aggregate the data to weekly revenue (`freq="W"`) and compare the accuracy of the weekly sums.
5. Implement the business rule "no revenue on Saturdays" and measure the effect on WAPE.
6. Add the number of distinct customers per day of the whole shop as a lagged regressor (`m.add_lagged_regressor`).